In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 5 - Week 7
# --------------------------------------------------
# The Week 7 .npy files contain the original data
# plus Weeks 1-6 exactly once.
#
# Function 5 has outputs on a much larger scale than
# the other functions, so Y is standardised for GP fitting.
#
# The raw objective is still treated as a maximisation problem.

In [2]:
X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 4

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y_raw = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y_raw)

X shape: (26, 4)
Y shape: (26,)

Current best observed input: [0.949995 1.       1.       1.      ]
Current best observed output: 7786.29242354451


In [3]:
y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (Y - y_mean) / y_std

best_y = Y_scaled[best_idx]

print("\nY mean:", y_mean)
print("Y std:", y_std)
print("Best scaled Y:", best_y)


Y mean: 1141.1629350004164
Y std: 2024.393091287392
Best scaled Y: 3.2825292267314503


In [4]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y_scaled)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
2.62**2 * Matern(length_scale=[1.87, 1.5, 1.8, 2], nu=2.5) + WhiteKernel(noise_level=0.0174)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [1.87088745 1.50445872 1.80433359 2.        ]
Normalised inverse-lengthscale sensitivity: [0.23719778 0.2949701  0.24594695 0.22188517]


In [6]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.1 0.1 0.1 0.1]
Wider search widths: [0.2 0.2 0.2 0.2]


In [7]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(50000, 4)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(20000, 4)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(10000, 4)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 80000


In [8]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.008]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 79362


In [9]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [10]:
mu_raw = mu * y_std + y_mean
sigma_raw = sigma * y_std

In [11]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [12]:
print("\nEI calibration:\n")

for xi in [0.0, 0.01, 0.05, 0.10]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n predicted raw mean =", round(mu_raw[idx], 3),
        "\n predicted raw std =", round(sigma_raw[idx], 3),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 EI = 0.07147133 

xi=0.01 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 EI = 0.0667977 

xi=0.05 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 EI = 0.05014796 

xi=0.1 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 EI = 0.03373304 



In [13]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n predicted raw mean =", round(mu_raw[idx], 3),
        "\n predicted raw std =", round(sigma_raw[idx], 3),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 UCB = 3.291009 

beta=0.25 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 UCB = 3.319858 

beta=0.5 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 UCB = 3.36794 

beta=1.0 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 UCB = 3.464105 

beta=1.5 
 candidate = [1. 1. 1. 1.] 
 predicted raw mean = 7764.523 
 predicted raw std = 389.349 
 UCB = 3.560269 



In [14]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("predicted raw mean =", mu_raw[mean_idx])
print("predicted raw std =", sigma_raw[mean_idx])


Highest predicted mean:
candidate = [1. 1. 1. 1.]
predicted raw mean = 7764.523166973404
predicted raw std = 389.34947787083763


In [15]:
# --------------------------------------------------
# Final Function 5 Week 7 selection
# --------------------------------------------------
#
# Function 5 outputs were standardised before GP fitting because their
# numerical scale is much larger than the other functions.
#
# The GP was trained on all accumulated observations and candidate-search
# widths were derived from the fitted ARD Matern lengthscales.
#
# All tested Expected Improvement xi values selected [1, 1, 1, 1].
# All tested UCB beta values also selected [1, 1, 1, 1].
# The same point had the highest GP predicted mean.
#
# This gives strong agreement across the surrogate and acquisition
# diagnostics, so the Week 7 point is selected without manually forcing
# any dimensions to the boundary.

xi = 0.0

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi
)

final_idx = np.argmax(EI)
week7_candidate = candidates[final_idx]

print("Week 7 Function 5 candidate:")
print(week7_candidate)

print("\nPredicted raw mean:", mu_raw[final_idx])
print("Predicted raw std:", sigma_raw[final_idx])
print("Expected Improvement:", EI[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 5 candidate:
[1. 1. 1. 1.]

Predicted raw mean: 7764.523166973404
Predicted raw std: 389.34947787083763
Expected Improvement: 0.07147132932597246

Portal format:
1.000000-1.000000-1.000000-1.000000
